In [6]:
%pip uninstall neo4j -y
%pip install "neo4j>=5.0"

Found existing installation: neo4j 4.2.1
Uninstalling neo4j-4.2.1:
  Successfully uninstalled neo4j-4.2.1
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [17]:
import sys

# buang semua modul neo4j yang ter-cache di memori
for name in [m for m in sys.modules if m == "neo4j" or m.startswith("neo4j.")]:
    del sys.modules[name]

import neo4j
print(neo4j.__version__)
from neo4j import GraphDatabase, RoutingControl
print("OK")

6.4.1
OK


In [19]:
import sys
print(sys.executable)

C:\Anaconda\envs\uts-gsd\python.exe


In [23]:
import sys, neo4j
print(sys.executable)
print(neo4j.__version__)
from neo4j import GraphDatabase, RoutingControl
print("OK")

C:\Anaconda\envs\uts-gsd\python.exe
6.4.1
OK


In [26]:
from getpass import getpass
from neo4j import GraphDatabase

USER = input("Username [neo4j]: ") or "neo4j"
PWD = getpass("Password: ")
print("panjang password:", len(PWD))  # untuk memastikan jumlah karakternya sesuai

try:
    with GraphDatabase.driver("neo4j://localhost:7687", auth=(USER, PWD)) as d:
        d.verify_connectivity()
    print("Login berhasil")
except Exception as e:
    print(type(e).__name__)

Username [neo4j]:  neo4j
Password:  ········


panjang password: 11
ClientError


In [28]:
from getpass import getpass
from neo4j import GraphDatabase, RoutingControl

URI = "neo4j://localhost:7687"
AUTH = ("neo4j", getpass("Password Neo4j: "))
DB = "nomor1"

def add_friend(driver, name, friend_name):
    driver.execute_query(
        "MERGE (a:Person {name: $name}) "
        "MERGE (friend:Person {name: $friend_name}) "
        "MERGE (a)-[:KNOWS]->(friend)",
        name=name, friend_name=friend_name, database_=DB,
    )

def print_friends(driver, name):
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
        "RETURN friend.name ORDER BY friend.name",
        name=name, database_=DB, routing_=RoutingControl.READ,
    )
    for record in records:
        print(record["friend.name"])

# database=DB penting: mencegah driver mencari home database "neo4j"
with GraphDatabase.driver(URI, auth=AUTH, database=DB) as driver:
    driver.verify_connectivity()
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    print_friends(driver, "Arthur")

Password Neo4j:  ········


Guinevere
Lancelot
Merlin


In [30]:
with GraphDatabase.driver(URI, auth=AUTH, database=DB) as driver:
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[r:KNOWS]->(b:Person) RETURN a.name AS dari, type(r) AS relasi, b.name AS ke",
        database_=DB, routing_=RoutingControl.READ,
    )
    for r in records:
        print(r["dari"], "-", r["relasi"], "->", r["ke"])

Arthur - KNOWS -> Guinevere
Arthur - KNOWS -> Lancelot
Arthur - KNOWS -> Merlin
